### Ollama + LlamaIndex CSV 챗봇

로컬 Ollama 모델을 사용해서 `ChatbotData.csv`를 검색 기반 챗봇으로 만듭니다.

In [12]:
# 처음 한 번만 실행하세요.
# !pip install llama-index-llms-ollama llama-index-embeddings-ollama
# !ollama pull nomic-embed-text
# !ollama pull llama3.2:3b
# superGamma4는 임베딩을 지원하지 않으므로 답변 생성 LLM으로만 사용합니다.

In [13]:
import time
from pathlib import Path

import pandas as pd
import requests
from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex, load_index_from_storage
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

In [14]:
CSV_CANDIDATES = [
    Path("../Data/pdf_sample1/ChatbotData.csv"),
    Path("Data/pdf_sample1/ChatbotData.csv"),
    Path("../Data/ChatbotData.csv"),
    Path("Data/ChatbotData.csv"),
]
CSV_PATH = next((path for path in CSV_CANDIDATES if path.exists()), CSV_CANDIDATES[0])

OLLAMA_BASE_URL = "http://localhost:11434"
DEFAULT_LLM_MODEL = "hf.co/NidAll/supergemma4-e4b-abliterated-Q4_K_M-GGUF:Q4_K_M"
OLLAMA_EMBED_MODEL = "nomic-embed-text"

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
    response.raise_for_status()
    installed_models = [model["name"] for model in response.json().get("models", [])]
except Exception as exc:
    raise RuntimeError("Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve` 또는 Ollama 앱을 실행하세요.") from exc


def has_model(model_name: str) -> bool:
    return any(name == model_name or name.startswith(f"{model_name}:") for name in installed_models)


if has_model(DEFAULT_LLM_MODEL):
    OLLAMA_LLM_MODEL = DEFAULT_LLM_MODEL
elif installed_models:
    OLLAMA_LLM_MODEL = installed_models[0]
    print(f"'{DEFAULT_LLM_MODEL}' 모델이 없어 설치된 모델로 대체합니다: {OLLAMA_LLM_MODEL}")
else:
    raise RuntimeError("설치된 Ollama 모델이 없습니다. 터미널에서 `ollama pull llama3.2:3b`를 실행하세요.")

if not has_model(OLLAMA_EMBED_MODEL):
    raise RuntimeError("임베딩 모델이 없습니다. 터미널에서 `ollama pull nomic-embed-text`를 실행하세요.")

print("CSV 경로:", CSV_PATH.resolve())
print("설치된 Ollama 모델:", installed_models)
print("사용할 LLM 모델:", OLLAMA_LLM_MODEL)
print("사용할 임베딩 모델:", OLLAMA_EMBED_MODEL)

CSV 경로: /Users/cheng80/Documents/WorkSpace/RAG/Data/pdf_sample1/ChatbotData.csv
설치된 Ollama 모델: ['llama3.2:3b', 'nomic-embed-text:latest', 'llama3.1:latest', 'hf.co/NidAll/supergemma4-e4b-abliterated-Q4_K_M-GGUF:Q4_K_M']
사용할 LLM 모델: hf.co/NidAll/supergemma4-e4b-abliterated-Q4_K_M-GGUF:Q4_K_M
사용할 임베딩 모델: nomic-embed-text


In [15]:
Settings.llm = Ollama(
    model=OLLAMA_LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    request_timeout=120.0,
)

Settings.embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

print("Ollama / LlamaIndex 설정 완료")

Ollama / LlamaIndex 설정 완료


#### CSV를 문서 형태로 변환

In [16]:
df = pd.read_csv(CSV_PATH)
df.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [17]:
TEXT_COLUMNS = ["Q", "A"]
METADATA_COLUMNS = ["label"]

MAX_ROWS = 200
df = df.head(MAX_ROWS).copy()

display(df.head())
print("문서와 대상 컬럼:", TEXT_COLUMNS)
print("메타데이터 컬럼:", METADATA_COLUMNS)
print("사용할 행 수:", len(df))

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


문서와 대상 컬럼: ['Q', 'A']
메타데이터 컬럼: ['label']
사용할 행 수: 200


In [18]:
def row_to_document(row: pd.Series, row_number: int) -> Document:
    text_parts = []

    for column in TEXT_COLUMNS:
        value = row[column]

        if pd.isna(value):
            continue

        text_parts.append(f"{column}: {value}")

    metadata = {
        "row_number": row_number,
        "label": row["label"],
    }

    return Document(
        text=" | ".join(text_parts),
        metadata=metadata,
    )


documents = [row_to_document(row, idx) for idx, row in df.iterrows()]

print("생성된 Document 수:", len(documents))
print("첫 번째 Document 예제")
print(documents[0].text)

생성된 Document 수: 200
첫 번째 Document 예제
Q: 12시 땡! | A: 하루가 또 가네요.


In [19]:
PERSIST_DIR = Path("storage/chatbot_ollama")
REBUILD_INDEX = False

if PERSIST_DIR.exists() and not REBUILD_INDEX:
    storage_context = StorageContext.from_defaults(persist_dir=str(PERSIST_DIR))
    index = load_index_from_storage(storage_context)
    print(f"저장된 인덱스를 불러왔습니다: {PERSIST_DIR}")
else:
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=str(PERSIST_DIR))
    print(f"새 인덱스를 만들고 저장했습니다: {PERSIST_DIR}")

chat_engine = index.as_chat_engine(
    chat_mode="context",
    similarity_top_k=3,
    verbose=False,
)

print("챗봇 준비 완료")

저장된 인덱스를 불러왔습니다: storage/chatbot_ollama
챗봇 준비 완료


In [ ]:
score_question = "12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?"
retriever = index.as_retriever(similarity_top_k=3)
retrieved_nodes = retriever.retrieve(score_question)

print("질문:", score_question)
print("\n[검색된 노드 스코어]")
for i, node_with_score in enumerate(retrieved_nodes, start=1):
    print(f"[{i}] score: {node_with_score.score:.6f}")
    print("text:", node_with_score.node.text)
    print("metadata:", node_with_score.node.metadata)
    print("-" * 60)

response = chat_engine.chat(score_question)

print("\n[응답]")
print(response)
print("\n[응답 source_nodes 스코어]")
for i, source_node in enumerate(response.source_nodes, start=1):
    print(f"[{i}] score: {source_node.score:.6f}")
    print("text:", source_node.node.text)
    print("metadata:", source_node.node.metadata)
    print("-" * 60)

In [20]:
question = "12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?"
start_time = time.perf_counter()
response = chat_engine.chat(question)
elapsed_time = time.perf_counter() - start_time

print("질문:", question)
print("응답:")
print(response)
print(f"출력 시간: {elapsed_time:.2f}초")

질문: 12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?
응답:
12시 땡!이라는 질문에는 **"하루가 또 가네요."** 라는 답변이 연결되어 있습니다.
출력 시간: 9.00초


In [21]:
while True:
    user_question = input("질문을 입력하세요: ").strip()

    if user_question.lower() in {"exit", "quit"}:
        print("챗봇을 종료합니다.")
        break

    if not user_question:
        print("빈 질문은 처리할 수 없습니다. 다시 입력해주세요.")
        continue

    start_time = time.perf_counter()
    answer = chat_engine.chat(user_question)
    elapsed_time = time.perf_counter() - start_time
    print("\n[응답]")
    print(answer)
    print(f"출력 시간: {elapsed_time:.2f}초")
    print("-" * 60)


[응답]
하루가 또 가네요.
출력 시간: 0.78초
------------------------------------------------------------

[응답]
제공해주신 정보(row_number, label, Q, A)만으로는 **'12시 땡!'이라는 질문 자체**가 이전에 제시된 4가지 예시(row_number 73, 194, 76, 그리고 사용자가 추가로 요청한 '12시 땡!'에 대한 답변)에 포함되어 있는지 여부만 알 수 있습니다.

**질문:** "여기 없는 질문은?"

이 질문은 **'내가 지금 제시된 예시 목록(혹은 이 대화창에 있는 내용)에 없는 질문이 무엇인가?'** 라는 의미로 해석됩니다.

만약 질문의 의도가 **'제공된 4개의 예시(Q: 같이 게임하자고 해도 되나? / Q: 고양이 동영상 보는 중 / Q: 같이 수영장 가기로 했어 / Q: 12시 땡!) 외에, 시스템이 알고 있는 다른 질문은 무엇인가?'** 라면, 저는 현재 제공된 데이터 셋을 기반으로 답변할 수 있습니다.

**하지만 가장 정확한 답변을 위해서는 '어떤 범위'에서 '없는 질문'을 찾으시는지 명확히 해주시면 좋습니다.**

---

**[참고: 현재 제공된 데이터 목록]**
1. **Q: 같이 게임하자고 해도 되나?** (A: 안 될 것도 없죠.)
2. **Q: 고양이 동영상 보는 중** (A: 완전 귀엽죠?)
3. **Q: 같이 수영장 가기로 했어** (A: 즐거운 시간 보내고 오세요!)
4. **Q: 12시 땡!** (A: 하루가 또 가네요.)
출력 시간: 27.81초
------------------------------------------------------------
챗봇을 종료합니다.
